In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from joblib import load, dump
import warnings

warnings.filterwarnings("ignore")

In [6]:
# Load the data from CSV files
popular_users = pd.read_csv("data/popular_users.csv")
ratings = pd.read_csv("data/ratings.csv")
movies = pd.read_csv('found_metadata.csv')

# Filter the popular users and their ratings
# Keep only the first 1000 unique popular users
filtered_users = popular_users.drop_duplicates('user').iloc[0:1000]

# Filter ratings to include only those from popular users
filtered_ratings = ratings.merge(filtered_users, left_on="user", right_on="user")

# Filter the movies to include only those present in filtered ratings
movies = movies[movies['Title'].isin(filtered_ratings['Title'].unique())]

# Select only the necessary columns from the movies dataset
selected_columns = ['Title', 'Year', 'Rated', 'Released', 'Runtime', 'Genre', 
                    'Director', 'Writer', 'Actors', 'Plot', 'Language', 'Country']
movies = movies[selected_columns]

# Define a reusable function to split and clean string lists (handling NaN values)
def split_and_strip(column_value):
    # Check if the value is a string; otherwise, return an empty list
    if isinstance(column_value, str):
        return [item.strip() for item in column_value.split(',')]
    return []

# Apply the split_and_strip function to relevant columns
columns_to_split = ['Genre', 'Director', 'Writer', 'Actors', 'Language', 'Country']
for column in columns_to_split:
    movies[f'{column} List'] = movies[column].apply(split_and_strip)

# Select the cleaned and relevant columns for the final movies DataFrame
final_columns = ['Title', 'Year', 'Rated', 'Runtime', 'Genre List', 'Director List', 
                 'Writer List', 'Actors List', 'Plot', 'Language List', 'Country List']
movies = movies[final_columns]

# Prepare the ratings DataFrame with user and movie IDs
# Factorize user and movie IDs to create unique integer IDs
df = filtered_ratings[['user', 'Title', 'Rating']]
df['user_id'] = pd.factorize(df['user'])[0]

# Assign movie IDs based on factorization of movie titles
movies['movie_id'] = pd.factorize(movies['Title'])[0]

# Create a mapping from movie titles to their corresponding movie IDs
title_to_id_map = dict(zip(movies['Title'], movies['movie_id']))

# Map movie IDs to the ratings DataFrame
df['movie_id'] = df['Title'].map(title_to_id_map)

# Merge ratings with movie metadata
merged_df = df.merge(movies, on='movie_id', how='left')

# Select and rename relevant columns in the ratings DataFrame
ratings = merged_df[['user', 'user_id', 'movie_id', 'Title_x', 'Rating', 'Genre List', 
                     'Director List', 'Writer List', 'Actors List', 'Plot', 'Language List', 'Country List']]
ratings = ratings.rename(columns={'Title_x': 'Title'}, inplace=False)

# Clean up the movies DataFrame by filling NaN values and dropping unnecessary columns
movies.fillna("", inplace=True)
#movies = movies.drop(movies.columns[[0]], axis=1)

# Drop the original user column from the ratings DataFrame
ratings = ratings.drop(columns=['user'])

# Min-max normalization to scale ratings between 0 and 1
min_rating = ratings['Rating'].min()
max_rating = ratings['Rating'].max()
ratings['normalized_rating'] = (ratings['Rating'] - min_rating) / (max_rating - min_rating)

In [7]:
# Load the SBERT model for encoding textual movie features
model = SentenceTransformer('all-MiniLM-L6-v2')

# Concatenate important text features into a single column
# Combine genres, directors, writers, actors, plot, language, and country to create meaningful textual input for SBERT
movies['combined_text'] = (
    movies['Genre List'].astype(str) + ' ' + 
    movies['Director List'].astype(str) + ' ' + 
    movies['Writer List'].astype(str) + ' ' + 
    movies['Actors List'].astype(str) + ' ' + 
    movies['Plot'].astype(str) + ' ' + 
    movies['Language List'].astype(str) + ' ' + 
    movies['Country List'].astype(str)
)

# Encode the combined text using SBERT
# Generate dense embeddings representing the semantic meaning of the combined text features
movie_embeddings = model.encode(movies['combined_text'].tolist(), show_progress_bar=True)

# Save embeddings as a pickle file
dump(movie_embeddings, 'sbert_embeddings.pkl')

# Store SBERT embeddings in the movies DataFrame
# Convert the embeddings into a list format and store them as a new column
movies['sbert_embedding'] = list(movie_embeddings)

# Merge SBERT embeddings with user ratings
# Join the SBERT-encoded movie representations with the user ratings DataFrame based on movie_id
ratings = ratings.merge(movies[['movie_id', 'sbert_embedding']], on='movie_id')

Batches:   0%|          | 0/731 [00:00<?, ?it/s]

In [8]:
# Function to compute a user's profile using the weighted average of SBERT embeddings
def compute_user_profile_sbert(user_id: int, rated_movies: pd.DataFrame = ratings) -> np.ndarray:
    # Filter movies rated by the given user
    user_movies = rated_movies[rated_movies['user_id'] == user_id]

    # Stack the SBERT embeddings of the user's rated movies into a matrix
    feature_vectors = np.vstack(user_movies['sbert_embedding'].values)

    # Retrieve the user's normalized ratings and reshape for weighting
    ratings = user_movies['normalized_rating'].values.reshape(-1, 1)

    # Compute the weighted average of the SBERT embeddings to generate the user profile
    user_profile = np.average(feature_vectors, axis=0, weights=ratings.flatten())
    return user_profile


In [ ]:
# Function to generate movie recommendations based on a user's profile

def gen_recs(user_id: int, rated_movies: pd.DataFrame = ratings, num_recs: int = 30) -> pd.DataFrame:
    # Compute the user's profile using SBERT-encoded features
    user_profile = compute_user_profile_sbert(user_id, rated_movies)

    # Compute cosine similarity between the user profile and all movie embeddings
    similarities = cosine_similarity(user_profile.reshape(1, -1), movie_embeddings).flatten()

    # Assign similarity scores to the movies DataFrame
    movies['similarity'] = similarities

    # Identify and exclude movies the user has already rated
    seen_movies = rated_movies[rated_movies['user_id'] == user_id]['movie_id'].tolist()
    recommended_movies = movies[~movies['movie_id'].isin(seen_movies)].copy().reset_index()

    # Rank the recommended movies by their similarity scores
    recommended_movies['similarity'] = similarities[recommended_movies.index]
    recommended_movies = recommended_movies.sort_values(by='similarity', ascending=False)

    # Return the top N recommendations with their titles and similarity scores
    return recommended_movies[['Title', 'similarity']].head(num_recs)

# Generate recommendations for user 3
gen_recs(3)

,Title,similarity
16245,In This Our Life,0.903184
18568,Xanadu,0.901710
12704,Underdogs,0.894323
6258,Infinity Baby,0.886877
9490,Constantine: City of Demons - The Movie,0.886788
14272,The Human Voice,0.886716
13239,The Front Page,0.885898
20737,One Child Nation,0.884836
20296,By the Time It Gets Dark,0.880938
9902,The Bloodhound,0.880341


In [23]:
# Function to compute the most similar movies based on the SBERT embeddings
def get_similar_movies(title: str, num_recs: int = 10) -> pd.DataFrame:
    # Step 1: Retrieve the SBERT embedding for the given movie
    target_embedding = movies.loc[movies['Title'] == title, 'sbert_embedding'].values[0]

    # Step 2: Compute cosine similarity between the target movie and all other movies
    similarities = cosine_similarity(np.array(target_embedding).reshape(1, -1), np.vstack(movies['sbert_embedding'].values)).flatten()

    # Step 3: Assign similarity scores to the movies DataFrame
    movies['similarity'] = similarities

    # Step 4: Exclude the target movie from the recommendations
    recommended_movies = movies[movies['Title'] != title].copy()

    # Step 5: Rank the recommended movies by their similarity scores
    recommended_movies = recommended_movies.sort_values(by='similarity', ascending=False)

    # Step 6: Return the top N recommendations with their titles and similarity scores
    return recommended_movies[['Title', 'similarity']].head(num_recs)

# Example: Get the top 10 movies similar to a given movie (e.g., movie_id = 101)
print(get_similar_movies(title="Nosferatu"))

                                                   Title  similarity
18330                                            Drácula    0.818258
2364                    Dracula Has Risen from the Grave    0.817983
19073                                   Blood of Dracula    0.815078
3494                               Bram Stoker's Dracula    0.809207
17173                          Bloodstone: Subspecies II    0.796557
10417                          Blood of Dracula's Castle    0.795015
17038                       Snow White: A Tale of Terror    0.790752
15581  Humanist Vampire Seeking Consenting Suicidal P...    0.790309
20587                                    Red Riding Hood    0.788731
20948                                    The Vampire Bat    0.787797
